## Fact_Sales

In [0]:
%sql

MERGE INTO adb_adventureworks_project.gold.fact_sales AS target
USING (
    SELECT
        s.OrderNumber,
        s.OrderLineItem,
        s.OrderDate,
        s.StockDate,
        s.ProductKey,
        s.CustomerKey,
        s.TerritoryKey,
        s.OrderQuantity,

        CAST(p.ProductPrice AS DECIMAL(18,4)) AS UnitPrice,
        CAST(p.ProductCost AS DECIMAL(18,4)) AS UnitCost,

        CAST(s.OrderQuantity * p.ProductPrice AS DECIMAL(18,4)) AS EstimatedRevenue,
        CAST(s.OrderQuantity * p.ProductCost AS DECIMAL(18,4)) AS EstimatedTotalCost,

        CAST(s.OrderQuantity * (p.ProductPrice - p.ProductCost)AS DECIMAL(18,4)) AS EstimatedGrossProfit

    FROM 
        adb_adventureworks_project.silver.sales_silver_table s
    INNER JOIN 
        adb_adventureworks_project.silver.products_silver_table p
    ON 
        s.ProductKey = p.ProductKey
) AS source

ON target.OrderNumber = source.OrderNumber
AND target.OrderLineItem = source.OrderLineItem

WHEN MATCHED AND (
    NOT (target.OrderDate <=> source.OrderDate)                 -- null-safe equality operator
    OR NOT (target.StockDate <=> source.StockDate)
    OR NOT (target.ProductKey <=> source.ProductKey)
    OR NOT (target.CustomerKey <=> source.CustomerKey)
    OR NOT (target.TerritoryKey <=> source.TerritoryKey)
    OR NOT (target.OrderQuantity <=> source.OrderQuantity)
    OR NOT (target.UnitPrice <=> source.UnitPrice)
    OR NOT (target.UnitCost <=> source.UnitCost)
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


In [0]:
%sql


SELECT
    'fact_sales' AS TableName,
    ROUND(SUM(EstimatedRevenue), 2) AS TotalRevenue
FROM adb_adventureworks_project.gold.fact_sales

UNION ALL

SELECT
    'agg_monthly_sales' AS TableName,
    ROUND(SUM(EstimatedRevenue), 2) AS TotalRevenue
FROM adb_adventureworks_project.gold.agg_monthly_sales;

